# Wake-Word Dataset Generator for Prosthetic Hand Control  
**Author:** Joaquín Cerdá-Boluda  
**Version:** 1.0  
**Last update:** 10/11/2025  
[GitHub Repository](https://github.com/ximocerda/ProstheticHand-VoiceCommands) ---

## Introduction

This notebook provides a complete and reproducible pipeline for generating a **synthetic voice-command dataset** specifically tailored for keyword spotting (KWS) and voice-driven control of a prosthetic hand.

Instead of relying on proprietary APIs or large-scale human recordings, this workflow is built entirely on **open-source components** and runs seamlessly in Google Colab. Speech synthesis is performed using the HuggingFace **MMS-TTS English** model, and additional variability is introduced through controlled acoustic transformations — pitch shifting, time stretching, and noise injection. These augmentations increase diversity and improve robustness while maintaining full reproducibility and minimal external dependencies.

The generator produces four functional categories of data:

- **hand grab** – wake-word class representing the grasp/closing command  
- **hand release** – complementary wake-word class for the opening command  
- **other words** – distractor phrases used to improve model generalization  
- **background** – synthetic noise segments used as negative examples  

All audio samples are normalized to **16 kHz**, **mono**, and a fixed **2-second duration**, ensuring compatibility with keyword-spotting architectures commonly used in embedded systems, TinyML frameworks, TensorFlow Lite, and EdgeImpulse pipelines.

This notebook serves as both a technical reference and a practical tool for generating custom datasets for speech-driven prosthetic-hand control.  
By running the cells, the full dataset will be automatically created inside the `dataset/` directory.

---


# Install and Setup

Install

In [ ]:
!apt-get -y install -qq ffmpeg
!pip install --quiet transformers librosa soundfile pydub numpy
print("✅ Environment ready.")


✅ Environment ready.


In [ ]:
from transformers import pipeline

# Load small English MMS-TTS model (stable on CPU/Colab)
tts = pipeline(
    task="text-to-speech",
    model="facebook/mms-tts-eng",
    device="cpu"
)

print("✅ MMS-TTS model loaded successfully.")


✅ MMS-TTS model loaded successfully.


In [ ]:
import numpy as np

TARGET_SR = 16000  # 16 kHz
DURATION_S = 2.0
DURATION_SAMPLES = int(TARGET_SR * DURATION_S)

def synthesize_base(text):
    """
    Robust version: handles dict, list-of-dicts, and unexpected outputs.
    Always returns a 1D float32 NumPy array or raises clear error.
    """
    output = tts(text)

    # Case: dict (normal MMS-TTS output)
    if isinstance(output, dict):
        y = output.get("audio", None)

    # Case: list containing dict(s)
    elif isinstance(output, list) and len(output) > 0:
        first = output[0]
        if isinstance(first, dict):
            y = first.get("audio", None)
        else:
            raise ValueError(f"Unexpected structure inside list: {type(first)}")

    else:
        raise ValueError(f"Unexpected TTS output type: {type(output)}")

    if y is None:
        raise ValueError(f"MMS-TTS returned no audio for text '{text}'. Output={output}")

    # Convert to float32 NumPy array
    y = np.asarray(y, dtype=np.float32)

    # Flatten 2D (usually (1, N)) → 1D
    if y.ndim > 1:
        y = y.flatten()

    return y

print("✅ Base synthesis function ready.")


✅ Base synthesis function ready.


In [ ]:
import librosa
import numpy as np

def augment_audio(y):
    """
    Robust augmentation: always returns a valid 1D float32 ndarray.
    Simulates speaker variability by applying soft formant-proxy distortions
    (pitch shift, temporal stretching, additive noise).
    """
    # Ensure proper dtype/shape
    y = np.asarray(y, dtype=np.float32).flatten()

    # Random pitch shift (-4 to +4 semitones)
    try:
        if np.random.rand() < 0.5 and len(y) > 512:
            steps = float(np.random.uniform(-4, 4))  # expanded pitch range
            y = librosa.effects.pitch_shift(y, sr=TARGET_SR, n_steps=steps).astype(np.float32)
    except Exception:
        pass

    # Random time stretch (0.9–1.1)
    try:
        if np.random.rand() < 0.5 and len(y) > 1024:
            rate = float(np.random.uniform(0.9, 1.1))
            y = librosa.effects.time_stretch(y, rate)
            y = np.asarray(y, dtype=np.float32).flatten()
    except Exception:
        pass

    # Additive Gaussian noise
    try:
        if np.random.rand() < 0.5:
            noise = np.random.normal(0, 0.02, size=y.shape).astype(np.float32)
            y = (y + noise).astype(np.float32)
    except Exception:
        pass

    # Final guard
    if y is None or not isinstance(y, np.ndarray) or y.ndim != 1:
        y = np.zeros(DURATION_SAMPLES, dtype=np.float32)

    return y.astype(np.float32)



In [ ]:
def normalize_length(y):
    assert y is not None, "normalize_length received None"
    y = np.asarray(y, dtype=np.float32).flatten()
    if len(y) < DURATION_SAMPLES:
        y = np.pad(y, (0, DURATION_SAMPLES - len(y)))
    else:
        y = y[:DURATION_SAMPLES]
    return y


In [ ]:
import soundfile as sf
import numpy as np

def synthesize_and_save(text, out_path):
    """
    Full pipeline with robust guards:
      1) base synthesis
      2) augmentation (with fallback)
      3) normalize length
      4) save WAV
    Never passes None through the pipeline.
    """
    # Step 1: base synthesis
    y = synthesize_base(text)
    if y is None:
        # Hard fallback: generate tiny silence (shouldn't happen now)
        y = np.zeros(DURATION_SAMPLES, dtype=np.float32)

    # Ensure proper array
    y = np.asarray(y, dtype=np.float32).flatten()

    # Step 2: augmentation with guard
    try:
        y = augment_audio(y)
        if y is None:
            raise ValueError("augment_audio returned None")
    except Exception:
        # Fallback: no augmentation
        y = np.asarray(y, dtype=np.float32).flatten()

    # Step 3: normalize length (extra guard)
    if y is None or not isinstance(y, np.ndarray):
        y = np.zeros(DURATION_SAMPLES, dtype=np.float32)

    y = normalize_length(y)

    # Step 4: save
    sf.write(out_path, y, TARGET_SR)

    return out_path


In [ ]:
import os

# Dataset root
root = "dataset"

# Class folders
classes = ["hand_grab", "hand_release", "background", "other_words"]

# Create directories
for c in classes:
    os.makedirs(f"{root}/{c}", exist_ok=True)

print("✅ Folder structure ready.")

# Number of samples per category
N_WAKE = 100       # hand grab & hand release
N_BG = 100         # background noise
N_OTHER = 100      # other words


# ✅ Generate 'hand grab' samples
print("🔄 Generating 'hand grab' samples...")
for i in range(N_WAKE):
    out_path = f"{root}/hand_grab/hand_grab_{i:04d}.wav"
    synthesize_and_save("hand grab", out_path)
print(f"✅ Generated {N_WAKE} samples for 'hand grab'.")


# ✅ Generate 'hand release' samples
print("🔄 Generating 'hand release' samples...")
for i in range(N_WAKE):
    out_path = f"{root}/hand_release/hand_release_{i:04d}.wav"
    synthesize_and_save("hand release", out_path)
print(f"✅ Generated {N_WAKE} samples for 'hand release'.")


# ✅ Generate 'background' samples
print("🔄 Generating 'background' samples...")

for i in range(N_BG):
    # Technique: generate silence + noise only
    silence = np.zeros(DURATION_SAMPLES, dtype=np.float32)
    noise = np.random.normal(0, 0.02, size=DURATION_SAMPLES).astype(np.float32)
    y = silence + noise

    out_path = f"{root}/background/bg_{i:04d}.wav"
    sf.write(out_path, y, TARGET_SR)

print(f"✅ Generated {N_BG} background noise samples.")


# ✅ Generate 'other words' samples
print("🔄 Generating 'other words' samples...")

other_phrases = [
    "hello", "go", "stop", "open hand", "close hand",
    "move forward", "grab it", "release it", "good job", "okay"
]

for i in range(N_OTHER):
    phrase = np.random.choice(other_phrases)
    out_path = f"{root}/other_words/other_{i:04d}.wav"
    synthesize_and_save(phrase, out_path)

print(f"✅ Generated {N_OTHER} 'other words' samples.")

print("\n🎉 DATASET COMPLETE — all files saved in 'dataset/' folder 🎉")


✅ Folder structure ready.
🔄 Generating 'hand grab' samples...
✅ Generated 100 samples for 'hand grab'.
🔄 Generating 'hand release' samples...
✅ Generated 100 samples for 'hand release'.
🔄 Generating 'background' samples...
✅ Generated 100 background noise samples.
🔄 Generating 'other words' samples...
✅ Generated 100 'other words' samples.

🎉 DATASET COMPLETE — all files saved in 'dataset/' folder 🎉
